In [1]:
import pandas as pd

df_control = pd.read_csv("df_control.csv")
df_test = pd.read_csv("df_test.csv")

In [4]:
df_control.info()

<class 'pandas.DataFrame'>
RangeIndex: 23532 entries, 0 to 23531
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   client_id         23532 non-null  int64  
 1   clnt_tenure_yr    23526 non-null  float64
 2   clnt_tenure_mnth  23526 non-null  float64
 3   clnt_age          23526 non-null  float64
 4   gendr             23526 non-null  str    
 5   num_accts         23526 non-null  float64
 6   bal               23526 non-null  float64
 7   calls_6_mnth      23526 non-null  float64
 8   logons_6_mnth     23526 non-null  float64
 9   variation         23532 non-null  str    
 10  visitor_id        23532 non-null  str    
 11  visit_id          23532 non-null  str    
 12  process_step      23532 non-null  str    
 13  date_time         23532 non-null  str    
dtypes: float64(7), int64(1), str(6)
memory usage: 2.5 MB


In [9]:
df_test.head(5)

,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,variation,visitor_id,visit_id,process_step,date_time
0,555,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0,Test,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34
1,647,12.0,151.0,57.5,M,2.0,30525.80,0.0,4.0,Test,66758770_53988066587,40369564_40101682850_311847,confirm,2017-04-12 15:47:45
2,934,9.0,109.0,51.0,F,2.0,32522.88,0.0,3.0,Test,810392784_45004760546,7076463_57954418406_971348,start,2017-04-18 02:38:52
3,1336,48.0,576.0,42.0,M,4.0,130537.18,6.0,9.0,Test,920624746_32603333901,614001770_19101025926_112779,confirm,2017-05-08 08:23:00
4,1346,14.0,177.0,46.0,F,2.0,822512.91,3.0,6.0,Test,123474046_4204671056,27144337_83739845380_214282,step_3,2017-06-06 18:28:43


In [10]:
# converting the "date_time" into a datetime type
df_test["date_time"] = pd.to_datetime(df_test["date_time"])
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 26968 entries, 0 to 26967
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   client_id         26968 non-null  int64         
 1   clnt_tenure_yr    26961 non-null  float64       
 2   clnt_tenure_mnth  26961 non-null  float64       
 3   clnt_age          26961 non-null  float64       
 4   gendr             26961 non-null  str           
 5   num_accts         26961 non-null  float64       
 6   bal               26961 non-null  float64       
 7   calls_6_mnth      26961 non-null  float64       
 8   logons_6_mnth     26961 non-null  float64       
 9   variation         26968 non-null  str           
 10  visitor_id        26968 non-null  str           
 11  visit_id          26968 non-null  str           
 12  process_step      26968 non-null  str           
 13  date_time         26968 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(7),

In [23]:
df_test['process_step'].unique()

<StringArray>
['confirm', 'step_3', 'step_1', 'start', 'step_2']
Length: 5, dtype: str

In [36]:
df_test = df_test.copy()

# Sort properly
df_test = df_test.sort_values(["visit_id", "process_step", "date_time"])

# Compute time between steps
df_test["time_spent"] = df_test.groupby("visit_id")["date_time"].diff().dt.total_seconds()

# Keeping only the  last occurrence per step
df_test = df_test.drop_duplicates(subset=["visit_id", "process_step"], keep="last")

# Pivot
df_pivot = df_test.pivot(index="visit_id", columns="process_step", values="time_spent")

# Sorting the process steps
step_order = ["start", "step_1", "step_2", "step_3", "confirm"]

df_test["process_step"] = pd.Categorical(df_test["process_step"], categories=step_order, ordered=True)

df_test = df_test.sort_values(["visit_id", "process_step", "date_time"])

In [37]:
from scipy.stats import f_oneway

f_stat, p_value = f_oneway(
    df_pivot["step_1"],
    df_pivot["step_2"],
    df_pivot["step_3"],
    df_pivot["confirm"]
)

print("Test group p-value:", p_value)

if p_value < 0.05:
    print("→ Completion times differ across steps")
else:
    print("→ No significant difference across steps")

Test group p-value: nan
→ No significant difference across steps


In [13]:
df_control["date_time"] = pd.to_datetime(df_control["date_time"])
df_control.info()

<class 'pandas.DataFrame'>
RangeIndex: 23532 entries, 0 to 23531
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   client_id         23532 non-null  int64         
 1   clnt_tenure_yr    23526 non-null  float64       
 2   clnt_tenure_mnth  23526 non-null  float64       
 3   clnt_age          23526 non-null  float64       
 4   gendr             23526 non-null  str           
 5   num_accts         23526 non-null  float64       
 6   bal               23526 non-null  float64       
 7   calls_6_mnth      23526 non-null  float64       
 8   logons_6_mnth     23526 non-null  float64       
 9   variation         23532 non-null  str           
 10  visitor_id        23532 non-null  str           
 11  visit_id          23532 non-null  str           
 12  process_step      23532 non-null  str           
 13  date_time         23532 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(7),